In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, avg, col
import os
import shutil
import time
import pandas as pd

spark = SparkSession.builder \
    .appName("Pertemuan10-StructuredStreaming") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Mengurangi jumlah partisi shuffle default (200) agar pemrosesan pada VM/laptop kita lebih cepat
# -- semakin kecil dataset (seperti simulasi kita), semakin tidak perlu partisi sebanyak itu
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 14:17:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/12 14:17:33 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession siap. Versi Spark: 3.5.9


In [2]:
# Persiapan ulang untuk latihan
spark = SparkSession.builder.appName("Latihan10").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

folder_latihan = "latihan10_stream"
shutil.rmtree(folder_latihan, ignore_errors=True)
os.makedirs(folder_latihan, exist_ok=True)

skema_latihan = StructType([
    StructField("trx_id", StringType(), True),
    StructField("kategori", StringType(), True),
    StructField("pendapatan", IntegerType(), True),
])
print("Siap untuk latihan. Folder:", folder_latihan)

Siap untuk latihan. Folder: latihan10_stream


In [3]:
# Definisikan readStream dari folder latihan10_stream
df_latihan = (
    spark.readStream
    .format("csv")
    .schema(skema_latihan)
    .option("header", True)
    .load(folder_latihan)
)

agg_latihan = df_latihan.groupBy("kategori").agg(
    avg("pendapatan").alias("rata_rata_pendapatan")
)

query = (
    agg_latihan.writeStream
    .outputMode("complete")
    .format("memory")
    .queryName("latihan_agg")
    .start()
)

In [4]:
def kirim_batch(df, folder, nama_file):
    # tulis ke berkas sementara (diawali "_" sehingga diabaikan Spark), lalu rename
    tmp = os.path.join(folder, f"_{nama_file}.tmp")
    df.to_csv(tmp, index=False)
    os.rename(tmp, os.path.join(folder, nama_file))

In [8]:
# Kirim (tulis) sebuah berkas CSV berisi minimal 3 baris data ke folder latihan10_stream
batch_1 = pd.DataFrame({
    "trx_id":     ["T1", "T2", "T3"],
    "kategori":   ["Elektronik", "Fashion", "Elektronik"],
    "pendapatan": [100000, 250000, 200000],
})
kirim_batch(batch_1, folder_latihan, "batch_1.csv")

time.sleep(3)batch_2 = pd.DataFrame({
    "trx_id":     ["T4", "T5"],
    "kategori":   ["Elektronik", "Makanan"],
    "pendapatan": [400000, 50000],
})
kirim_batch(batch_2, folder_latihan, "batch_2.csv")

time.sleep(3)
query.processAllAvailable()

spark.sql("""
    SELECT kategori, ROUND(rata_rata_pendapatan, 2) AS rata_rata_pendapatan
    FROM latihan_agg ORDER BY kategori
""").show()
query.processAllAvailable()

spark.sql("""
    SELECT kategori, ROUND(rata_rata_pendapatan, 2) AS rata_rata_pendapatan
    FROM latihan_agg ORDER BY kategori
""").show()

+----------+--------------------+
|  kategori|rata_rata_pendapatan|
+----------+--------------------+
|Elektronik|            150000.0|
|   Fashion|            250000.0|
+----------+--------------------+



In [10]:
# Kirim batch kedua dengan kategori yang sama seperti salah satu kategori pada batch pertama
batch_2 = pd.DataFrame({
    "trx_id":     ["T4", "T5"],
    "kategori":   ["Elektronik", "Makanan"],
    "pendapatan": [400000, 50000],
})
kirim_batch(batch_2, folder_latihan, "batch_2.csv")

time.sleep(3)
query.processAllAvailable()

spark.sql("""
    SELECT kategori, ROUND(rata_rata_pendapatan, 2) AS rata_rata_pendapatan
    FROM latihan_agg ORDER BY kategori
""").show()

+----------+--------------------+
|  kategori|rata_rata_pendapatan|
+----------+--------------------+
|Elektronik|           233333.33|
|   Fashion|            250000.0|
|   Makanan|             50000.0|
+----------+--------------------+



Apakah nilai rata-ratanya berubah sesuai perhitungan gabungan kedua batch? Ya, rata-rata Elektronik berubah dari 150.000 menjadi 233.333,33,

Mengapa Spark Structured Streaming mewajibkan skema didefinisikan secara eksplisit (StructType), sementara pemrosesan batch pada Pertemuan 4 bisa memakai inferSchema=True? Pada batch, seluruh data sudah tersedia sehingga Spark dapat memindai datanya lebih dulu untuk menebak tipe tiap kolom. Pada streaming, data belum semuanya ada dan terus berdatangan, jadi tidak ada “data lengkap” untuk dipindai; menebak dari berkas pertama saja berisiko salah dan dapat merusak query bila berkas berikutnya bertipe lain. Selain itu, query streaming berjalan lama dan menyimpan state/checkpoint, sehingga skemanya harus tetap dan konsisten sejak awal.

In [13]:
query.stop()
spark.stop()